# 📊 Bases de Datos ENAHO 2024-2025
## Documentación técnica completa

**Última actualización:** Octubre 2026  
**Versión:** v5 (corregida)  
**Responsable:** Dafne  
**Proyecto:** Investigación económica — Uso de billeteras digitales y acceso a crédito formal en Perú

---

## 📑 Índice

1. [Resumen ejecutivo](#1-resumen-ejecutivo)
2. [Estructura de archivos](#2-estructura-de-archivos)
3. [Fuentes de datos (ENAHO)](#3-fuentes-de-datos-enaho)
4. [Descripción de cada base](#4-descripción-de-cada-base)
5. [Diccionario de variables](#5-diccionario-de-variables)
6. [Construcción de variables clave](#6-construcción-de-variables-clave)
7. [Estadísticas descriptivas](#7-estadísticas-descriptivas)
8. [Uso en Stata](#8-uso-en-stata)
9. [Verificación y calidad](#9-verificación-y-calidad)
10. [Limitaciones](#10-limitaciones)

---

## 1. Resumen ejecutivo

Este proyecto construye **7 archivos CSV** que empalman los datos de la Encuesta Nacional de Hogares (ENAHO) del INEI para los años **2024 y 2025**, permitiendo analizar:

- **H1**: ¿El uso de billeteras digitales (Yape/Plin) aumenta la probabilidad de acceder a crédito formal?
- **H2**: ¿Este efecto difiere entre trabajadores formales e informales?
- **H3**: ¿El uso de billeteras digitales se relaciona con el uso de mecanismos financieros informales?

**Novedades de la versión v5:**
- ✅ Medida ordinal de intensidad de uso de billetera (0-4)
- ✅ Desagregación de crédito formal en tarjeta vs préstamo
- ✅ Control de conectividad (internet hogar)
- ✅ **Mecanismo informal amplio corregido** (junta + ahorro con familiares + ahorro en casa + préstamo informal) — **G8 excluido por ser ahorro FORMAL**
- ✅ Dos factores de expansión: hogar (FACTOR07) y empleo (FAC500A)
- ✅ Dos niveles de análisis: jefes de hogar e individuos 14+
- ✅ Separación clara entre `relacion_parentesco` (crudo 1-11) y `es_jefe_hogar` (dummy 0/1)

---

## 2. Estructura de archivos

Los 7 archivos se generan en cascada. Cada uno depende del anterior:

```
┌──────────────────────────────────────────────────────────┐
│  NIVEL 1: MASTERS (datos crudos por año)                 │
│  ├── MASTER_2024.csv      (117,721 × 91)                 │
│  └── MASTER_2025.csv      (115,145 × 91)                 │
└──────────────────────────────────────────────────────────┘
                            ↓
┌──────────────────────────────────────────────────────────┐
│  NIVEL 2: PANEL (empalme 2024-2025)                      │
│  └── PANEL_2024_2025.csv  (32,079 × 47)                  │
└──────────────────────────────────────────────────────────┘
                            ↓
        ┌───────────────────┴───────────────────┐
        ↓                                       ↓
┌─────────────────────────┐       ┌─────────────────────────┐
│  NIVEL 3: REGRESIONES   │       │  NIVEL 4: H3            │
│  (sin crédito 2024)     │       │  (todos los ocupados)   │
│  ├── JEFES (main)       │       │  ├── H3_JEFES (robustez)│
│  └── INDIVIDUOS (rob.)  │       │  └── H3_INDIV (main)    │
└─────────────────────────┘       └─────────────────────────┘
```

### Tabla resumen

| # | Archivo | Filas | Columnas | Uso |
|---|---|---|---|---|
| 1 | `MASTER_2024.csv` | 117,721 | 91 | Intermedio |
| 2 | `MASTER_2025.csv` | 115,145 | 91 | Intermedio |
| 3 | `PANEL_2024_2025.csv` | 32,079 | 47 | Intermedio |
| 4 | `BASE_REGRESIONES_JEFES.csv` | 6,358 | 45 | **Main H1/H2** |
| 5 | `BASE_REGRESIONES_INDIVIDUOS.csv` | 14,696 | 45 | Robustez H1/H2 |
| 6 | `BASE_H3_JEFES.csv` | 7,708 | 52 | Robustez H3 |
| 7 | `BASE_H3_INDIVIDUOS.csv` | 17,273 | 52 | **Main H3** |

---

## 3. Fuentes de datos (ENAHO)

Los datos provienen del INEI. Se usan 5 módulos por año:

| Módulo | Nombre | Qué aporta |
|---|---|---|
| **Módulo 100** | Vivienda y hogar | Dominio, estrato, factor hogar, internet del hogar |
| **Módulo 200** | Miembros del hogar | Sexo, edad, relación de parentesco |
| **Módulo 300** | Educación | Nivel educativo alcanzado |
| **Módulo 500** | Empleo e ingresos | Ocupación, informalidad, billetera, crédito, barreras, factor empleo |
| **Sumaria** | Resumen del hogar | Ingreso total, gasto total, número de miembros |

### Archivos fuente ENAHO

```
ENAHO/
├── 966-Modulo01/Enaho01-2024-100.csv       (2024)
├── 966-Modulo02/Enaho01-2024-200.csv
├── 966-Modulo03/Enaho01A-2024-300.csv
├── 966-Modulo05/Enaho01a-2024-500.csv
├── 966-Modulo34/Sumaria-2024-12g.csv
├── 1031-Modulo01/Enaho01-2025-100.csv      (2025)
├── 1031-Modulo02/Enaho01-2025-200.csv
├── 1031-Modulo03/Enaho01A-2025-300.csv
├── 1031-Modulo05/Enaho01a-2025-500.csv
└── 1031-Modulo34/Sumaria-2025-12g.csv
```

---

## 4. Descripción de cada base

### 4.1 `MASTER_2024.csv` / `MASTER_2025.csv`

**Naturaleza:** Base cruda anual, sin filtros.

**Contenido:** Todos los individuos encuestados, con los 5 módulos ya mergeados por `CONGLOME-VIVIENDA-HOGAR-CODPERSO`.

**¿Para qué sirve?** Es la base intermedia. No se usa directamente en Stata; se usa para construir el panel.

**Filtros:** Ninguno.

**Tamaño:**
- 2024: 117,721 filas × 91 columnas
- 2025: 115,145 filas × 91 columnas

---

### 4.2 `PANEL_2024_2025.csv`

**Naturaleza:** Panel empalmado (personas en ambos años).

**Contenido:** Solo las personas que aparecen en 2024 **Y** en 2025.

**¿Para qué sirve?** Es la base que permite el análisis longitudinal.

**Filtros aplicados:**
- Estar en 2024 **Y** 2025 (empalme por `llave_persona`)

**Tamaño:** 32,079 filas × 47 columnas

**Columnas de interés:**
- Variables de 2024: `CreditoFormal_2024`, `IntensidadBilletera`, `ingreso_percapita_2024`, `InternetHogar`
- Variables de 2025: `CreditoFormal_2025`, `CreditoInformal_2025`, `MecanismoInformalAmplio_2025`, `razon_no_credito`
- Factores: `FACTOR07` (hogar) y `FAC500A` (empleo)

---

### 4.3 `BASE_REGRESIONES_JEFES.csv` ⭐

**Naturaleza:** Muestra analítica principal (H1 y H2).

**Contenido:** Jefes de hogar ocupados que **no tenían crédito formal en 2024**.

**Filtros aplicados (en cascada):**

| Paso | Filtro | Filas restantes |
|---|---|---|
| 0 | Panel | 32,079 |
| 1 | Excluir P301A==99 (sin nivel educativo) | 32,052 |
| 2 | Solo jefes de hogar (P203==1) | 9,090 |
| 3 | Solo ocupados (P507 no missing) | 7,708 |
| 4 | Sin crédito formal 2024 | 6,358 |
| 5 | Sin missing en variables clave | 6,358 |

**Uso:** Probit H1 y H2 (main).

**Tamaño final:** 6,358 filas × 45 columnas

**Variable dependiente:** `nuevo_credito_formal` (588 casos, 9.25%)

---

### 4.4 `BASE_REGRESIONES_INDIVIDUOS.csv` ⭐

**Naturaleza:** Robustez de H1 y H2.

**Contenido:** Todos los individuos de 14+ años, ocupados, sin crédito formal 2024.

**Filtros aplicados:**

| Paso | Filtro | Filas restantes |
|---|---|---|
| 0 | Panel | 32,079 |
| 1 | Edad ≥ 14 | 23,879 |
| 2 | Excluir P301A==99 | 23,879 |
| 3 | Solo ocupados | 17,273 |
| 4 | Sin crédito formal 2024 | 14,696 |
| 5 | Sin missing | 14,696 |

**Uso:** Robustez de H1 y H2.

**Tamaño final:** 14,696 filas × 45 columnas

---

### 4.5 `BASE_H3_JEFES.csv`

**Naturaleza:** Muestra analítica para H3 (jefes).

**Contenido:** Jefes de hogar ocupados **sin importar si tenían crédito formal en 2024**.

**Filtros aplicados:**

| Paso | Filtro | Filas restantes |
|---|---|---|
| 0 | Panel | 32,079 |
| 1 | Excluir P301A==99 | 32,052 |
| 2 | Solo jefes de hogar | 9,090 |
| 3 | Solo ocupados | 7,708 |
| 4 | **NO se filtra crédito formal 2024** | 7,708 |
| 5 | Sin missing | 7,708 |

**Uso:** Robustez de H3.

**Tamaño final:** 7,708 filas × 52 columnas

**Variable dependiente:** `mecanismo_informal_amplio_2025` (1,712 casos)

---

### 4.6 `BASE_H3_INDIVIDUOS.csv` ⭐

**Naturaleza:** Muestra analítica principal para H3.

**Contenido:** Todos los individuos 14+, ocupados, sin importar crédito formal 2024.

**Filtros aplicados:**

| Paso | Filtro | Filas restantes |
|---|---|---|
| 0 | Panel | 32,079 |
| 1 | Edad ≥ 14 | 23,879 |
| 2 | Excluir P301A==99 | 23,879 |
| 3 | Solo ocupados | 17,273 |
| 4 | **NO se filtra crédito formal 2024** | 17,273 |
| 5 | Sin missing | 17,273 |

**Uso:** Análisis principal de H3 (más N, más potencia).

**Tamaño final:** 17,273 filas × 52 columnas

**Variable dependiente:** `mecanismo_informal_amplio_2025` (3,308 casos)

---

## 5. Diccionario de variables

### 5.1 Identificación

| Variable | Tipo | Descripción | Fuente ENAHO |
|---|---|---|---|
| `id_persona` | texto | Llave única de la persona (CONGLOME-VIVIENDA-HOGAR-CODPERSO) | Construida |
| `conglomerado` | numérica | Conglomerado (para clustering) | CONGLOME |

### 5.2 Características personales

| Variable | Tipo | Descripción | Fuente ENAHO |
|---|---|---|---|
| `relacion_parentesco` | 1-11 | Relación con el jefe (1=jefe, 2=esposo, 3=hijo, etc.) | P203 |
| `es_jefe_hogar` | 0/1 | Dummy: 1 si es jefe de hogar | (P203==1) |
| `sexo` | 1/2 | 1 = hombre, 2 = mujer | P207 |
| `edad` | años | Edad en años cumplidos | P208A |
| `nivel_educativo` | 1-12 | Nivel educativo alcanzado | P301A |
| `miembros_hogar` | numérica | Número de miembros del hogar | MIEPERHO |

### 5.3 Ubicación

| Variable | Tipo | Descripción | Fuente ENAHO |
|---|---|---|---|
| `dominio` | 1-8 | Dominio geográfico (Costa Norte, ..., Lima) | DOMINIO |
| `estrato` | 1-8 | Estrato geográfico (urbano/rural por tamaño) | ESTRATO |

### 5.4 Empleo

| Variable | Tipo | Descripción | Construcción |
|---|---|---|---|
| `ocupado` | 0/1 | 1 = tiene ocupación | P507 no missing |
| `trabajador_informal` | 0/1 | 1 = informal | Ver sección 6.1 |

### 5.5 Billetera digital (TRATAMIENTO) ⭐

| Variable | Tipo | Descripción |
|---|---|---|
| `tiene_billetera` | 0/1 | Tiene Yape/Plin (P558E1_8==8) |
| `usa_billetera` | 0/1 | Usa billetera en al menos 1 rubro |
| `n_categorias_billetera` | 0-12 | En cuántos de 12 rubros la usa |
| `intensidad_billetera` | 0-4 | Ordinal: no tiene / tiene pero no usa / baja / media / alta |

**Detalle de los 12 rubros** (variables P558H1_7 a P558H12_7):

| # | Rubro |
|---|---|
| 1 | Alimentos de pan llevar |
| 2 | Alimentos preparados |
| 3 | Productos de lavandería / aseo hogar |
| 4 | Servicios de la vivienda (luz, agua, celular) |
| 5 | Combustible para cocinar |
| 6 | Productos de aseo personal |
| 7 | Prendas de vestir y calzado |
| 8 | Muebles y enseres |
| 9 | Artefactos electrodomésticos |
| 10-12 | Otros |

### 5.6 Crédito formal (VARIABLE DEPENDIENTE) ⭐

⚠️ **Importante:** El diccionario ENAHO dice que:
- `P558E1_4 == 4` = **TARJETA DE CRÉDITO** (no préstamo)
- `P558E1_9 == 9` = **PRÉSTAMO** (no tarjeta)
- `P558E1_10 == 10` = **no tiene ninguno**

| Variable | Tipo | Descripción |
|---|---|---|
| `credito_formal_2024` | 0/1 | Tenía tarjeta O préstamo en 2024 |
| `credito_tarjeta_2024` | 0/1 | Específicamente tarjeta (2024) |
| `credito_prestamo_2024` | 0/1 | Específicamente préstamo (2024) |
| `credito_formal_2025` | 0/1 | Tiene tarjeta o préstamo en 2025 |
| `credito_tarjeta_2025` | 0/1 | Específicamente tarjeta (2025) |
| `credito_prestamo_2025` | 0/1 | Específicamente préstamo (2025) |
| `sin_credito_formal` | 0/1 | No tiene tarjeta ni préstamo |
| `nuevo_credito_formal` | 0/1 | Nuevo crédito en 2025 (VD principal) |
| `nuevo_credito_tarjeta` | 0/1 | Nueva tarjeta |
| `nuevo_credito_prestamo` | 0/1 | Nuevo préstamo |

### 5.7 Mecanismos financieros informales (SOLO en bases H3)

⚠️ **CORRECCIÓN v5:** `P558G8` NO es informal. Mide **ahorro EN un banco, financiera, caja, cooperativa** = ahorro **FORMAL**. NO forma parte del outcome.

| Variable | Tipo | Descripción | Fuente |
|---|---|---|---|
| `credito_informal_2025` | 0/1 | **Crédito informal estricto**: recibió préstamo de prestamista/familiar/conocido | P558G7==7 |
| `mecanismo_informal_amplio_2025` | 0/1 | **Mecanismo informal amplio**: junta + ahorro con familiares + ahorro en casa + préstamo informal | Construida |
| `usa_junta` | 0/1 | Ahorró en junta/pandero | P558G1==1 |
| `ahorro_con_familiares` | 0/1 | Ahorró guardando dinero con familiares o conocidos | P558G2==2 |
| `ahorro_en_casa` | 0/1 | Ahorró guardando dinero en su casa | P558G3==3 |
| `recibio_prestamo_informal` | 0/1 | Recibió préstamo de prestamista, familiar o conocido | P558G7==7 |
| `presto_dinero` | 0/1 | Prestó dinero a otros | P558G6==6 |
| `ahorro_en_banco` | 0/1 | Ahorró **EN banco/financiera/caja/cooperativa** ⚠️ **FORMAL** | P558G8==8 |

**Definición del MECANISMO INFORMAL AMPLIO:**  
`mecanismo_informal_amplio = 1` si cumple **al menos una** de:
- Usa junta/pandero (G1)
- Ahorra con familiares o conocidos (G2)
- Ahorra en su casa (G3)
- Recibe préstamo de prestamista/familiar/conocido (G7)

**NO incluye G8.**

### 5.8 Barreras de crédito (P558F1B)

⚠️ La pregunta solo se hace a quienes NO tienen préstamo ni tarjeta. NO usar como control en H1/H2.

| Variable | Tipo | Descripción |
|---|---|---|
| `razon_no_credito` | 6-12 | Código de razón principal de no tener crédito |
| `razon_demanda` | 0/1 | 6 = no necesita / no le interesa |
| `razon_deuda` | 0/1 | 7 = ya tiene deuda |
| `razon_intereses` | 0/1 | 8 = intereses muy altos |
| `razon_costos` | 0/1 | 9 = servicios costosos |
| `razon_requisitos` | 0/1 | 10 = requisitos que no tiene |
| `razon_infocorp` | 0/1 | 11 = está en INFOCORP |
| `razon_otro` | 0/1 | 12 = otro |
| `tiene_barrera_credito` | 0/1 | Cualquier razón ≠ 6 |

### 5.9 Ingreso y gasto

| Variable | Tipo | Descripción |
|---|---|---|
| `ingreso_percapita_2024` | numérica | Ingreso per cápita del hogar en 2024 |
| `gasto_percapita_2024` | numérica | Gasto per cápita del hogar en 2024 |
| `ingreso_percapita_2025` | numérica | Ingreso per cápita del hogar en 2025 |
| `gasto_percapita_2025` | numérica | Gasto per cápita del hogar en 2025 |
| `log_ingreso_2024` | numérica | ln(ingreso_percapita_2024) |

### 5.10 Conectividad (CONTROL)

| Variable | Tipo | Descripción | Fuente |
|---|---|---|---|
| `internet_hogar` | 0/1 | Hogar tiene conexión a internet | P1144==1 |

### 5.11 Ponderación

| Variable | Tipo | Descripción | Fuente |
|---|---|---|---|
| `factor_expansion_hogar` | numérica | Factor de expansión del hogar | FACTOR07 (Módulo 100) |
| `factor_expansion_empleo` | numérica | Factor de expansión de empleo/ingresos | FAC500A (Módulo 500) |

**Recomendación:**
- Para descriptivos de hogar: `factor_expansion_hogar`
- Para regresiones de empleo/crédito: **`factor_expansion_empleo`**

---

## 6. Construcción de variables clave

### 6.1 `trabajador_informal`

```python
informal = (
    (P507 == 5)                                   # TFNR
    | (P507 == 6)                                 # Trabajador del hogar
    | ((P507 == 2) & (P510A1 == 3))               # Independiente sin RUC
    | ((P507 in [3, 4]) & (P511A == 7))           # Dependiente sin contrato
)
```

### 6.2 `intensidad_billetera` (0-4)

```
Paso 1: tiene_billetera = 1 si P558E1_8 == 8
Paso 2: n_categorias_billetera = suma de las 12 dummies P558H{i}_7 == 7
Paso 3:
    intensidad = 0  si tiene_billetera == 0
    intensidad = 1  si tiene_billetera == 1 y n_categorias == 0
    intensidad = 2  si 1 <= n_categorias <= 3
    intensidad = 3  si 4 <= n_categorias <= 7
    intensidad = 4  si n_categorias >= 8
```

### 6.3 `nuevo_credito_formal`

```python
nuevo_credito_formal = 1 si credito_formal_2025 == 1
(ya se filtró credito_formal_2024 == 0)
```

### 6.4 `mecanismo_informal_amplio_2025`

```python
mecanismo_informal_amplio = 1 si AL MENOS UNA:
    P558G1 == 1  (junta/pandero)
    O P558G2 == 2  (ahorro con familiares)
    O P558G3 == 3  (ahorro en casa)
    O P558G7 == 7  (préstamo informal recibido)
```

**NO incluye G8** (que es ahorro formal).

---

## 7. Estadísticas descriptivas

### 7.1 Nuevo crédito formal (jefes)

| Categoría | N | % |
|---|---|---|
| Total | 588 | 9.25% |
| Solo tarjeta | 129 | 21.9% |
| Solo préstamo | 409 | 69.6% |
| Ambos | 50 | 8.5% |

### 7.2 Nuevo crédito formal (individuos)

| Categoría | N |
|---|---|
| Total | 1,161 |
| Solo tarjeta | 338 |
| Solo préstamo | 724 |
| Ambos | 99 |

### 7.3 H3 — Cruce clave (billetera × mecanismos informales)

| Nivel | Jefes | Individuos |
|---|---|---|
| Informales | 5,646 | 12,963 |
| Usan billetera (informales) | 670 | 1,883 |
| Con crédito informal puntual (G7) | 196 | 346 |
| Con mecanismo informal AMPLIO | 1,712 | 3,308 |
| **Billetera × informal puntual** | **26** | **53** |
| **Billetera × mecanismo AMPLIO** | **181** | **431** |

**Descomposición del mecanismo amplio:**

| Componente | Jefes | Individuos |
|---|---|---|
| `usa_junta` (G1) | 240 | 478 |
| `ahorro_con_familiares` (G2) | 27 | 69 |
| `ahorro_en_casa` (G3) | 1,308 | 2,540 |
| `recibio_prestamo_informal` (G7) | 196 | 346 |
| `presto_dinero` (G6) | 54 | 106 |
| `ahorro_en_banco` (G8, FORMAL) | 625 | 1,410 |

### 7.4 Distribución de intensidad de billetera (jefes)

| Nivel | N | % |
|---|---|---|
| 0 (no tiene) | 4,858 | 76.4% |
| 1 (tiene, no usa) | 628 | 9.9% |
| 2 (baja) | 515 | 8.1% |
| 3 (media) | 282 | 4.4% |
| 4 (alta) | 75 | 1.2% |

---

## 8. Uso en Stata

### 8.1 Cargar cada base

```stata
import delimited "BASE_REGRESIONES_JEFES.csv", clear encoding("UTF-8")
import delimited "BASE_REGRESIONES_INDIVIDUOS.csv", clear encoding("UTF-8")
import delimited "BASE_H3_JEFES.csv", clear encoding("UTF-8")
import delimited "BASE_H3_INDIVIDUOS.csv", clear encoding("UTF-8")
```

### 8.2 Crear variables derivadas

```stata
gen mujer = (sexo == 2)
gen urbano = (estrato <= 6)
```

### 8.3 Probit H1 con intensidad

```stata
probit nuevo_credito_formal i.intensidad_billetera ///
       edad c.edad#c.edad mujer i.nivel_educativo urbano ///
       miembros_hogar i.dominio log_ingreso_2024 ///
       trabajador_informal internet_hogar ///
       [pw=factor_expansion_empleo], vce(cluster conglomerado)

margins, dydx(intensidad_billetera)
```

### 8.4 Probit H2 con interacción

```stata
probit nuevo_credito_formal ///
       usa_billetera##trabajador_informal ///
       edad c.edad#c.edad mujer i.nivel_educativo urbano ///
       miembros_hogar i.dominio log_ingreso_2024 ///
       internet_hogar ///
       [pw=factor_expansion_empleo], vce(cluster conglomerado)

margins, dydx(usa_billetera) at(trabajador_informal=0)
margins, dydx(usa_billetera) at(trabajador_informal=1)
```

### 8.5 Probit H3 con outcome amplio

```stata
use "BASE_H3_INDIVIDUOS.dta", clear

probit mecanismo_informal_amplio_2025 usa_billetera ///
       edad c.edad#c.edad mujer i.nivel_educativo ///
       i.dominio log_ingreso_2024 ///
       [pw=factor_expansion_empleo], vce(cluster conglomerado)

margins, dydx(usa_billetera)
```

### 8.6 Heterogeneidad

```stata
probit nuevo_credito_formal ///
       c.usa_billetera##i.nivel_educativo ///
       edad c.edad#c.edad mujer urbano ///
       miembros_hogar i.dominio log_ingreso_2024 ///
       trabajador_informal internet_hogar ///
       [pw=factor_expansion_empleo], vce(cluster conglomerado)

probit nuevo_credito_formal ///
       c.usa_billetera##c.log_ingreso_2024 ///
       edad c.edad#c.edad mujer i.nivel_educativo urbano ///
       miembros_hogar i.dominio ///
       trabajador_informal internet_hogar ///
       [pw=factor_expansion_empleo], vce(cluster conglomerado)
```

---

## 9. Verificación y calidad

| Chequeo | Resultado |
|---|---|
| Las 4 bases existen | ✅ |
| 0 filas rotas | ✅ |
| 0 missing en variables críticas | ✅ |
| 5 niveles en `intensidad_billetera` | ✅ |
| Desagregación tarjeta/préstamo | ✅ |
| `mecanismo_informal_amplio` sin G8 | ✅ |
| Sin `internet_fijo_hogar` ni `internet_movil_prepago` | ✅ |

---

## 10. Limitaciones

### 10.1 Medición de intensidad
Es por extensión (número de rubros), no por frecuencia ni monto.

### 10.2 Potencia estadística en H3
- Jefes: 181 casos con billetera + mecanismo informal amplio
- Individuos: 431 casos
- Mejor que outcome puntual (26 / 53), pero errores estándar pueden ser grandes.

### 10.3 Unidad de análisis
Principal: jefe de hogar (H1/H2), individuo 14+ (H3).

### 10.4 Atrición del panel
Solo el 27% de las personas de 2024 se encuentran en 2025 (32,079 de 117,721). No corregido en esta versión.

### 10.5 Control de barreras de crédito
`tiene_barrera_credito` viene de `P558F1B`, que solo se pregunta a quienes no tienen crédito. NO usar como control en H1/H2.

### 10.6 Mecanismo informal amplio
El 76% del outcome amplio viene de `ahorro_en_casa` (G3), que es ahorro, no crédito. Reportar siempre junto al outcome puntual (`credito_informal_2025`).

---

## 📞 Contacto

Para dudas sobre el código: scripts `construir_bases_v5.py` y `construir_h3_v5.py`.

**Última revisión:** Octubre 2026